AI as a Service (Kubernetes Variante)
---------------

Beispiel für einen **SGLang-Client**, der mit einem **SGLang Inference Server auf einem separaten Rechner** kommuniziert.

In diesem Szenario läuft **SGLang auf Kubernetes** als zentral betriebener Inference-Service. Bevor der Client Anfragen senden kann, muss jedoch zunächst das gewünschte **Modell als eigener Pod beziehungsweise Container gestartet** werden. Erst danach steht es über eine **OpenAI-kompatible API** für die Clients zur Verfügung.

Im Unterschied zu Ollama unterstützt **SGLang pro Container in der Regel nur ein einzelnes LLM**. Sollen mehrere Modelle parallel betrieben werden, werden diese deshalb üblicherweise als **separate Deployments oder Pods** gestartet und über unterschiedliche Services, Endpoints oder Ports bereitgestellt.

Folgende Befehle in der `rPodman Sandbox` ausführen:

    kubectl create ns qwen-instruct
    kubectl apply -n qwen-instruct -f https://github.com/mc-b/lernvirt/raw/refs/heads/main/examples/aiaas/k8s/qwen-instruct-deployment.yaml
    kubectl apply -n qwen-instruct -f https://github.com/mc-b/lernvirt/raw/refs/heads/main/examples/aiaas/k8s/qwen-instruct-service.yaml

Und um den richtigen Port zu finden

    kubectl -n qwen-instruct get services


- - -

Der folgende Code zeigt, wie aus einem **Python-Jupyter-Notebook** über diese OpenAI-kompatible API auf einen betriebenen SGLang-Service zugegriffen wird.

Die Verbindung erfolgt über den API-Endpoint des Servers. Der verwendete API-Key dient dabei lediglich als Platzhalter.

Die Funktion `ask` kapselt einen einfachen Chat-Request. Da ein SGLang-Container typischerweise genau ein Modell bereitstellt, dient der Modellname hier in erster Linie der expliziten Adressierung des geladenen Dienstes beziehungsweise der Kompatibilität zur OpenAI-Schnittstelle. Der übrige Code muss dadurch nicht angepasst werden.

In [ ]:
from openai import OpenAI
import time

def ask(model, prompt, port=31868, max_tokens=2048):

    client = OpenAI(
        base_url=f"http://10.3.24.17:{port}/v1",
        api_key="sglang"
    )

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=max_tokens,
    )

    end = time.perf_counter()
    duration = end - start

    answer = response.choices[0].message.content

    prompt_tokens = response.usage.prompt_tokens
    completion_tokens = response.usage.completion_tokens
    total_tokens = response.usage.total_tokens

    tokens_per_sec = completion_tokens / duration

    print(f"Antwortzeit: {duration:.2f} Sekunden")
    print(f"Prompt Tokens: {prompt_tokens}")
    print(f"Completion Tokens: {completion_tokens}")
    print(f"Total Tokens: {total_tokens}")
    print(f"Decode Speed: {tokens_per_sec:.2f} tokens/s")

    return answer


Dieses Codebeispiel sendet eine Anfrage an das Sprachmodell `Qwen/Qwen2.5-0.5B-Instruct`, das auf einem **SGLang-Inference-Server** betrieben wird. Das kompakte Instruct-Modell eignet sich für einfache Dialoge, kurze Erklärungen und grundlegende technische Fragestellungen. Der Zugriff erfolgt über die **OpenAI-kompatible API** von SGLang.

Bei der Interpretation der gemessenen Antwortzeit ist zu beachten, dass der **erste Request nach dem Start eines Containers** oft langsamer ist. In dieser Phase werden Modellgewichte geladen und die Inferenz-Engine initialisiert. Dieser Vorgang wird als **Kaltstart** bezeichnet. Nachfolgende Anfragen sind meist schneller, da das Modell bereits im Speicher liegt (**Warm-Start**).

Im Unterschied zu Ollama betreibt **SGLang typischerweise nur ein Modell pro Container**. Mehrere Modelle werden daher meist über mehrere Container oder Instanzen betrieben, die über unterschiedliche Ports angesprochen werden.

Für Performance-Analysen lohnt es sich zusätzlich, **auf dem Server die Container-Logs zu betrachten**. Dort zeigt SGLang unter anderem den **Token-Durchsatz (Tokens pro Sekunde)** sowie weitere Laufzeitmetriken der Inferenz an.


In [ ]:
print(ask(
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Erkläre HTTP Codes kurz."
))

- - -
`HuggingFaceTB/SmolLM2-1.7B-Instruct` ist ein kompaktes Instruct-Sprachmodell aus der SmolLM2-Familie von Hugging Face. Mit rund 1.7 Milliarden Parametern gehört es zu den kleineren LLMs und ist darauf ausgelegt, auf moderater Hardware effizient zu laufen, etwa auf einer einzelnen GPU oder leistungsfähigen CPU-Systemen.

Das Modell wurde speziell für **dialogorientierte Aufgaben und instruktionbasierte Prompts** trainiert. Es eignet sich für kurze Erklärungen, einfache Programmierhilfe, Zusammenfassungen oder strukturierte Antworten auf technische Fragen. Durch seine geringe Modellgrösse reagiert es meist schnell und verursacht deutlich geringere Ressourcen- und Latenzanforderungen als grössere Modelle.

SmolLM2-Modelle werden häufig in **lokalen Inferenz-Setups, Edge-Umgebungen oder experimentellen LLM-Workflows** eingesetzt, bei denen ein guter Kompromiss zwischen Modellqualität, Geschwindigkeit und Hardwarebedarf wichtig ist.

In [ ]:
print(ask(
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Schreibe ein kurzes Python Beispiel für einen REST Client.",
    port=31838
))

- - -

### Kubernetes Zugriff 

Mit einem kleinen Trick kann Jupyter Lab direkt auf Kubernetes auf der DGX Spark zugreifen.

1. Datei config.txt anlegen
2. In der `rPodman Sandbox` - `cat /etc/rancher/k3s/k3s.yaml`
3. Inhalt in config.txt ablegen und `server:` Zeile auf WireGuard IP-Adresse ändern


In [ ]:
%%bash
kubectl --kubeconfig config.txt get all

Log Ausgabe

In [ ]:
%%bash
kubectl --kubeconfig config.txt logs deployment/qwen-sglang-model

In [ ]:
%%bash
kubectl --kubeconfig config.txt logs deployment/smollm2-sglang-model